# SeeSaw — Notebook 1: Data Preparation

Converts TinyStories + SeeSaw StoryBeatRecord exports into the Gemma 4 instruction format.

**Output:** `gs://seesaw-models/training-data/seesaw_beats_train.jsonl`

**Runtime:** Free Colab T4, ~30 minutes

See `docs/FINE_TUNING.md` for full details.

In [ ]:
# Step 1: Install dependencies
!pip install datasets google-cloud-storage

In [ ]:
# Step 2: Load TinyStories
from datasets import load_dataset

ds = load_dataset('roneneldan/TinyStories', split='train')
sample = ds.shuffle(seed=42).select(range(12000))  # oversample, filter below
print(f'Loaded {len(sample)} stories')

In [ ]:
# Step 3: Safety filter
BANNED_TERMS = [
    'kill', 'die', 'dead', 'blood', 'gun', 'knife', 'weapon',
    'scary', 'nightmare', 'horror', 'alcohol', 'drug', 'sex',
]

def is_safe(text: str) -> bool:
    text_lower = text.lower()
    return not any(term in text_lower for term in BANNED_TERMS)

safe = sample.filter(lambda x: is_safe(x['text']))
print(f'After safety filter: {len(safe)} stories')

In [ ]:
# Step 4: Convert to SeeSaw instruction format (Gemma chat template)
import json

SYSTEM_PROMPT = """You are SeeSaw, a gentle storytelling companion for children aged 4-8.
Generate story beats as JSON: {\"story_text\": \"...\", \"question\": \"...\", \"is_ending\": false}"""

def to_instruction_format(story_text: str) -> str:
    # Split into story + question heuristic
    sentences = [s.strip() for s in story_text.split('.') if s.strip()]
    if len(sentences) < 2:
        return None
    story = '. '.join(sentences[:3]) + '.'
    question = 'What do you think happens next?'
    payload = json.dumps({'story_text': story, 'question': question, 'is_ending': False})
    return (
        f'<bos><start_of_turn>user\n{SYSTEM_PROMPT}\n\nContinue the story.<end_of_turn>\n'
        f'<start_of_turn>model\n{payload}<end_of_turn>'
    )

examples = []
for row in safe:
    formatted = to_instruction_format(row['text'])
    if formatted:
        examples.append({'text': formatted})

# Limit to 8000
examples = examples[:8000]
print(f'Final dataset: {len(examples)} examples')

In [ ]:
# Step 5: Save to GCS
import json
from google.cloud import storage

GCS_BUCKET = 'seesaw-models'
BLOB_NAME  = 'training-data/seesaw_beats_train.jsonl'

jsonl = '\n'.join(json.dumps(ex) for ex in examples)

client = storage.Client()
bucket = client.bucket(GCS_BUCKET)
blob   = bucket.blob(BLOB_NAME)
blob.upload_from_string(jsonl, content_type='application/jsonl')

print(f'Uploaded {len(examples)} examples to gs://{GCS_BUCKET}/{BLOB_NAME}')